In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input,Embedding,Dense,LSTM,Dropout,Bidirectional
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer


import warnings
warnings.filterwarnings("ignore")

c:\Users\Amir sohail\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
df = pd.read_csv("news.csv")
df.head()

,text,label
0,Gere faults Trump for blurring meaning of 'ref...,1
1,German parties start to find common ground in ...,1
2,Senate Democratic leader says Attorney General...,1
3,"Tennis: Kyrgios fined $10,000 for Shanghai wal...",1
4,Trump Threw Mar-A-Lago Fundraiser For Woman A...,0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45757 entries, 0 to 45756
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    45757 non-null  object
 1   label   45757 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 715.1+ KB


In [4]:
df.shape

(45757, 2)

In [5]:
df.columns

Index(['text', 'label'], dtype='object')

In [6]:
df.isnull().sum()

text     0
label    0
dtype: int64

In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
def tolower(txt):
    return txt.lower()

def remove_emoj(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new+=i
    return new


def remove_punc(txt):
    import string
    return txt.translate(str.maketrans('','',string.punctuation))

def remove_num(txt):
    new = ""
    for i in txt:
        if not i.isdigit():
            new+=i
    return new

In [9]:
df["text"] = df["text"].apply(tolower)
df["text"] = df["text"].apply(remove_emoj)
df["text"] = df["text"].apply(remove_punc)
df["text"] = df["text"].apply(remove_num)

df.head()

,text,label
0,gere faults trump for blurring meaning of refu...,1
1,german parties start to find common ground in ...,1
2,senate democratic leader says attorney general...,1
3,tennis kyrgios fined for shanghai walk off te...,1
4,trump threw maralago fundraiser for woman at ...,0


In [10]:
Max_len = 100
Max_words = 10000
embedding_dim = 100

In [11]:
X = df.drop("label",axis=1)
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

In [12]:
y.unique()

array([1, 0])

In [13]:
tokenizer = Tokenizer(num_words=Max_words,oov_token="<OOV>")
tokenizer.fit_on_texts(X_train["text"])
train_seq = tokenizer.texts_to_sequences(X_train["text"])
train_pad = pad_sequences(train_seq,maxlen=Max_len,padding="post",truncating="post")
test_seq = tokenizer.texts_to_sequences(X_test["text"])
test_pad = pad_sequences(test_seq,maxlen=Max_len,padding="post",truncating="post")


Caching the list of root modules, please wait!
(This will only be done once - type '%rehashx' to reset cache!)


Caching the list of root modules, please wait!
(This will only be done once - type '%rehashx' to reset cache!)


Caching the list of root modules, please wait!
(This will only be done once - type '%rehashx' to reset cache!)



In [14]:
model_ann = Sequential([
    Embedding(input_dim=Max_words,output_dim=embedding_dim,input_length=Max_len),
    Dense(128,activation="relu"),
    Bidirectional(LSTM(64,return_sequences=True)),
    Dropout(0.4),
    Dense(128,activation="relu"),
    Bidirectional(LSTM(64,return_sequences=True)),
    Dropout(0.3),
    Dense(32,activation="relu"),
    Bidirectional(LSTM(16,return_sequences=False)),
    Dropout(0.2),
    Dense(32,activation="relu"),
    Dense(16,activation="relu"),
    Dense(8,activation="relu"),
    Dropout(0.1),
    Dense(1,activation="sigmoid"),
])

# model_ann = Sequential([
#     Embedding(input_dim=Max_words,output_dim=embedding_dim,input_length=Max_len),
#     Bidirectional(LSTM(64,return_sequences=True)),
#     Dropout(0.3),
#     Bidirectional(LSTM(32,return_sequences=False)),
#     Dropout(0.2),
#     Dense(64,activation="relu"),
#     Dense(32,activation="relu"),
#     Dense(1,activation="sigmoid")
# ])


In [15]:
model_ann.compile(loss="binary_crossentropy",optimizer="adam",metrics=["accuracy"])

In [16]:
his = model_ann.fit(train_pad,y_train,validation_data=(test_pad,y_test),epochs=2,batch_size=128)

Epoch 1/2
240/240 ━━━━━━━━━━━━━━━━━━━━ 227s 869ms/step - accuracy: 0.9147 - loss: 0.2300 - val_accuracy: 0.9730 - val_loss: 0.0875
Epoch 2/2
240/240 ━━━━━━━━━━━━━━━━━━━━ 192s 802ms/step - accuracy: 0.9818 - loss: 0.0661 - val_accuracy: 0.9825 - val_loss: 0.0639


In [17]:
from joblib import dump

model_ann.save("model_ann.h5")
dump(tokenizer,"tokenizer.pkl")
dump(X.columns.tolist(),"columns.pkl")

['columns.pkl']

In [18]:
X.columns.tolist()

['text']